# UdaPlay — Part 2: Multi-Agent Architecture

This notebook wires together the four agents (RAG, Validation, Web Search, and the Orchestrator) and runs the three sample test scenarios end-to-end, showing each step's reasoning trace and the final cited answer.

> Run `Udaplay_01_solution_project.ipynb` first (or `scripts/build_index.py`) so the FAISS index already exists on disk.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import verify_environment
from src.logging_config import get_logger

verify_environment()
logger = get_logger("notebook_02")


## 1. Inspect the individual tools

Each tool can be called directly (outside of an agent) for quick sanity checks.

In [ ]:
from src.tools.retrieve_game_tool import retrieve_game

print(retrieve_game.invoke({"query": "Pokémon Red"}))


In [ ]:
from src.tools.evaluate_retrieval_tool import evaluate_retrieval

sample_docs = retrieve_game.invoke({"query": "Pokémon Red"})
evaluation = evaluate_retrieval.invoke({
    "query": "When was Pokémon Red launched, and on what platform?",
    "retrieved_docs": sample_docs,
})
evaluation


In [ ]:
from src.tools.web_search_tool import game_web_search

print(game_web_search.invoke({"query": "What is Rockstar Games working on right now?"})[:800])


## 2. Build the Orchestrator (wires up RAG Agent, Validation Agent, and Web Search Agent)

In [ ]:
from src.agents.orchestrator_agent import OrchestratorAgent

orchestrator = OrchestratorAgent()


## 3. Run the three sample test scenarios

Each call logs every step (retrieval, evaluation, fallback decision, synthesis) to the console and to `logs/udaplay.log`.

### Query 1 — Internal DB Match

In [ ]:
answer_1 = orchestrator.ask("When was Pokémon Red launched, and on what platform?")
print(answer_1.to_display_string())


### Query 2 — Fallback Trigger (not in the internal database)

In [ ]:
answer_2 = orchestrator.ask("What is Rockstar Games working on right now?")
print(answer_2.to_display_string())


### Query 3 — Release Verification (also exercises the web fallback,
since the sample dataset only includes the original 2018 *God of War*, not *Ragnarok*)

In [ ]:
answer_3 = orchestrator.ask("When was God of War Ragnarok released?")
print(answer_3.to_display_string())


## 4. Multi-turn memory check

The orchestrator keeps a short-term conversation history across turns.

In [ ]:
for turn in orchestrator.state.history:
    print("Q:", turn.query)
    print("A:", turn.answer.answer[:150], "...")
    print("Source:", turn.answer.source)
    print()


Part 2 complete: all three sample scenarios ran through the full retrieve → evaluate → (optional fallback) → synthesize pipeline, each producing a structured, cited `FinalAnswer`, with a full reasoning trace available in `logs/udaplay.log`.